# Right heart catheterization and 30-day mortality
Causal machine learning on the SUPPORT data, following the eight-step pipeline.

In 1996 Connors et al. reported that intensive care patients who received a right heart catheter on their first day were more likely to die within 30 days. Randomized trials later found no such effect. This notebook estimates the effect again, asks how much unmeasured confounding it would take to close the gap to the trials, and checks whether the effect differs between patients.

Throughout, $W$ is `swang1`, equal to 1 if the patient received a catheter on day 1, and $Y$ is `dth30`, equal to 1 if the patient died within 30 days. $X$ are the patient characteristics recorded on day 1. The ATE $\tau = E[Y(1) - Y(0)]$ is a risk difference and is reported in percentage points, $\tau(x)$ is the CATE.

| Step | Content |
|---|---|
| 1. Research question and data | load the data, handle missing covariates, define the analysis sample |
| 2. Causal structure | DAG and backdoor check |
| 3. Estimand | ATE, overlap population, CATE |
| 4. Assumptions | SUTVA, overlap, balance |
| 5. Estimation | nuisance models, causal forest, estimator ladder, comparison with published estimates |
| 6. Evaluation | calibration test, heterogeneity check, RATE |
| 7. Robustness | placebo, random common cause, semi-synthetic outcome, sensitivity benchmarked against the RCTs |
| 8. Interpretation | best linear projection, partial dependence, summary |

Run the notebook with Restart Kernel and Run All. The first run downloads the data into `data/`.

Work in progress: setup and Steps 1.1 to 1.3 are done.

## 0. Setup

In [1]:
import hashlib
import json
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import networkx as nx
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

from econml.grf import CausalForest, RegressionForest

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

SEED = 42
N_TREES = 2000               # econml needs a multiple of 4 for inference
N_TREES_REFUTE = 1000        # lighter forests for the refits in Step 7
LEAF_GRID = [5, 10, 20, 40]  # candidate min_samples_leaf values for the causal forest
N_FOLDS = 5
ALPHA = 0.05
Z_CRIT = stats.norm.ppf(1 - ALPHA / 2)

# Published estimates, used for validation in Step 5 and as benchmark in Step 7
REF = {
    "connors_or": {"estimate": 1.24, "ci": (1.03, 1.49),
                   "source": "Connors et al. 1996, JAMA 276:889-897, propensity-matched OR for 30-day mortality"},
    "hirano_imbens_rd": {"range": (0.053, 0.062),
                         "source": "Hirano and Imbens 2001, Health Serv Outcomes Res Methodol 2:259-278, "
                                   "increase in 30-day mortality from weighting estimators"},
    "rct_meta_or": {"estimate": 1.04, "ci": (0.90, 1.20),
                    "source": "Shah et al. 2005, JAMA 294:1664-1670, meta-analysis of 13 RCTs with 5,051 patients"},
}

rng = np.random.default_rng(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR = ROOT / "data"
RESULTS = {}                 # key numbers, exported at the end

# Plot style: no top and right spines, no grid, one accent colour, source line under each figure
GREY, LIGHT, DARK = "#9A9A9A", "#D9D9D9", "#2B2B2B"
ACCENT, ACCENT_2, GOLD = "#0F6E56", "#C0502E", "#C9A227"
SOURCE = ("Source: SUPPORT study, RHC data from Vanderbilt Biostatistics, "
          "5,735 ICU patients, 1989 to 1994. Own analysis.")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 11,
    "axes.titlesize": 14, "axes.labelsize": 11, "axes.labelcolor": DARK,
    "xtick.color": DARK, "ytick.color": DARK, "axes.edgecolor": GREY,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.size": 0, "ytick.major.size": 0,
})


def titles(ax, title, subtitle=None):
    ax.set_title(title, loc="left", fontweight="bold", color=DARK, pad=24 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=GREY, fontsize=10, va="bottom")


def save(fig, name, source=SOURCE):
    fig.text(0.01, -0.02, source, fontsize=8, color=GREY, ha="left", va="top")
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()


def fmt_ci(est, lo, hi, unit=" pp"):
    return f"{est:.1f}{unit}, 95% CI {lo:.1f} to {hi:.1f}"

---
## Step 1. Research question and data

### 1.1 Research question
The main question is by how many percentage points a catheter on day 1 changes 30-day mortality.

Two further questions follow from it. The randomized trials found no effect on mortality, so how strong would an unmeasured confounder have to be to move our estimate to theirs? Severity of illness that physicians saw at the bedside but that is not recorded in the data is the obvious candidate. And does the effect differ between patients, for example by disease category or severity?

The 1996 result had consequences. An editorial called for a moratorium on the catheter, and its use in the US fell by about two thirds over the following decade. Whether the observational estimate shows real harm or residual confounding is therefore also a question about how far this kind of analysis can be trusted.

### 1.2 Data
The data come from SUPPORT, a study of seriously ill adults admitted to five US teaching hospitals between 1989 and 1994. Connors et al. analysed 5,735 of these patients from nine disease categories, and Vanderbilt Biostatistics publishes this subset for methods research on causal inference. The study is observational. Physicians decided who got a catheter, so the two groups can differ in ways that also affect survival.

All covariates refer to day 1, when the patient entered the study and the treatment decision was made. Discharge, death and last-contact dates and the 180-day death indicator come later and are not used as covariates.

In [2]:
# [CONCEPT] Download the data once, verify the checksum, then always read the local copy
# [WHY]     The checksum guarantees that every run analyses exactly the same file
DATA_URL = "https://hbiostat.org/data/repo/rhc.csv"
DATA_SHA256 = "9ef4ab578be4b40ad5d97d3a7e08ffdc1f9f76aeeefee51b4996e4221556f8e8"


def load_rhc(path=DATA_DIR / "rhc.csv"):
    if not path.exists():
        path.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(DATA_URL, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != DATA_SHA256:
        raise ValueError(f"Checksum mismatch for {path}. Delete the file and run the cell again.")
    return pd.read_csv(path, index_col=0)


df_raw = load_rhc()

W_COL, Y_COL = "swang1", "dth30"
# [CONCEPT] Variables measured after the treatment decision are never used as covariates
# [WHY]     They can be affected by the catheter itself, and adjusting for them would bias the effect
POST_COLS = ["dschdte", "dthdte", "lstctdte", "death", "t3d30"]
ID_COLS = ["ptid", "sadmdte"]

# Short labels for tables and figures. Source: https://hbiostat.org/data/repo/rhc.html
LABELS = {
    "swang1": "Right heart catheterization", "dth30": "Death within 30 days",
    "t3d30": "Days survived up to day 30", "death": "Death within 180 days",
    "sadmdte": "Study admission date", "dschdte": "Discharge date", "dthdte": "Date of death",
    "lstctdte": "Date of last contact", "ptid": "Patient ID",
    "age": "Age", "sex": "Sex", "race": "Race", "edu": "Years of education", "income": "Income",
    "ninsclas": "Medical insurance", "cat1": "Primary disease category", "cat2": "Secondary disease category",
    "resp": "Respiratory diagnosis", "card": "Cardiovascular diagnosis", "neuro": "Neurological diagnosis",
    "gastr": "Gastrointestinal diagnosis", "renal": "Renal diagnosis", "meta": "Metabolic diagnosis",
    "hema": "Hematologic diagnosis", "seps": "Sepsis diagnosis", "trauma": "Trauma diagnosis",
    "ortho": "Orthopedic diagnosis",
    "adld3p": "Activities of daily living", "das2d3pc": "Duke Activity Status Index",
    "dnr1": "DNR order on day 1", "ca": "Cancer", "surv2md1": "SUPPORT 2-month survival estimate",
    "aps1": "APACHE score", "scoma1": "Glasgow Coma Score", "wtkilo1": "Weight", "temp1": "Temperature",
    "meanbp1": "Mean blood pressure", "resp1": "Respiratory rate", "hrt1": "Heart rate",
    "pafi1": "PaO2/FiO2 ratio", "paco21": "PaCO2", "ph1": "pH", "wblc1": "White blood cell count",
    "hema1": "Hematocrit", "sod1": "Sodium", "pot1": "Potassium", "crea1": "Creatinine",
    "bili1": "Bilirubin", "alb1": "Albumin", "urin1": "Urine output",
    "cardiohx": "Comorbidity: cardiovascular", "chfhx": "Comorbidity: congestive heart failure",
    "dementhx": "Comorbidity: dementia, stroke or Parkinson's", "psychhx": "Comorbidity: psychiatric",
    "chrpulhx": "Comorbidity: chronic pulmonary", "renalhx": "Comorbidity: chronic renal",
    "liverhx": "Comorbidity: cirrhosis or hepatic failure", "gibledhx": "Comorbidity: upper GI bleeding",
    "malighx": "Comorbidity: malignancy", "immunhx": "Comorbidity: immunosuppression",
    "transhx": "Transfer from another hospital", "amihx": "Comorbidity: definite myocardial infarction",
}

# dth30 and t3d30 are not in the documentation, so we check their meaning against the dates
days_to_death = df_raw["dthdte"] - df_raw["sadmdte"]
assert ((df_raw[Y_COL] == "Yes") == (days_to_death <= 30)).all()
assert (df_raw["t3d30"] == days_to_death.clip(upper=30).fillna(30)).all()


def role(col):
    if col == W_COL:
        return "treatment"
    if col == Y_COL:
        return "outcome"
    if col in POST_COLS:
        return "post-treatment, not used"
    if col in ID_COLS:
        return "identifier, not used"
    return "candidate confounder"


data_dictionary = pd.DataFrame({
    "label": [LABELS[c] for c in df_raw.columns],
    "role": [role(c) for c in df_raw.columns],
    "missing_share": df_raw.isna().mean().round(3).to_numpy(),
}, index=df_raw.columns)

n_raw, n_rhc = len(df_raw), int((df_raw[W_COL] == "RHC").sum())
n_conf = int((data_dictionary["role"] == "candidate confounder").sum())
print(f"{n_raw:,} patients, {n_rhc:,} with a catheter, {n_raw - n_rhc:,} without")
print(f"Missing outcomes: {df_raw[Y_COL].isna().sum()}, candidate confounders: {n_conf}")
RESULTS.update({"n_raw": n_raw, "n_rhc_raw": n_rhc})
data_dictionary.sort_values(["role", "missing_share"], ascending=[True, False])

5,735 patients, 2,184 with a catheter, 3,551 without
Missing outcomes: 0, candidate confounders: 53


,label,role,missing_share
cat2,Secondary disease category,candidate confounder,0.791
adld3p,Activities of daily living,candidate confounder,0.749
urin1,Urine output,candidate confounder,0.528
cat1,Primary disease category,candidate confounder,0.000
ca,Cancer,candidate confounder,0.000
cardiohx,Comorbidity: cardiovascular,candidate confounder,0.000
chfhx,Comorbidity: congestive heart failure,candidate confounder,0.000
dementhx,"Comorbidity: dementia, stroke or Parkinson's",candidate confounder,0.000
psychhx,Comorbidity: psychiatric,candidate confounder,0.000
chrpulhx,Comorbidity: chronic pulmonary,candidate confounder,0.000


5,735 patients, 2,184 of them with a catheter. No outcome is missing, and 53 variables remain as candidate confounders. `missing_share` is the share of patients without a value for that variable.

### 1.3 Missing covariates
Three covariates have large gaps: the secondary disease category, activities of daily living and urine output. For each one the question is why the value is missing. If the gap itself carries information, it can be coded as a category. If it depends on what happened after day 1, neither the variable nor a missing indicator belongs in the adjustment set.

In [3]:
# [CONCEPT] Compare patients with and without a value on treatment rate and 30-day mortality
# [WHY]     If the gap goes along with the outcome, it may reflect the course after day 1
GAP_COLS = ["cat2", "adld3p", "urin1"]
treated = df_raw[W_COL] == "RHC"
died = df_raw[Y_COL] == "Yes"

rows = []
for col in GAP_COLS:
    miss = df_raw[col].isna()
    rows.append({"variable": LABELS[col], "missing_share": miss.mean(),
                 "rhc_if_missing": treated[miss].mean(), "rhc_if_observed": treated[~miss].mean(),
                 "mortality_if_missing": died[miss].mean(), "mortality_if_observed": died[~miss].mean()})
gap_table = pd.DataFrame(rows).set_index("variable")

# Share with an ADL value by survival time, share with a urine output value by admission year.
# t3d30 is only used here to describe the gaps, never as a covariate.
survival_group = pd.cut(df_raw["t3d30"].where(died, 31), [0, 7, 14, 30, 31],
                        labels=["died on day 2 to 7", "died on day 8 to 14", "died on day 15 to 30", "alive at day 30"])
adl_by_survival = df_raw["adld3p"].notna().groupby(survival_group, observed=True).mean().rename_axis(None)
admission_year = pd.to_datetime(df_raw["sadmdte"], unit="D", origin="1960-01-01").dt.year   # SAS dates
urine_by_year = df_raw["urin1"].notna().groupby(admission_year).mean().rename_axis(None)

print(gap_table.round(3), "\n")
print("Share with an ADL value, by survival:")
print(adl_by_survival.round(3).to_string(), "\n")
print("Share with a urine output value, by admission year:")
print(urine_by_year.round(3).to_string())

                            missing_share  rhc_if_missing  rhc_if_observed  mortality_if_missing  mortality_if_observed
variable                                                                                                               
Secondary disease category          0.791           0.369            0.427                 0.300                  0.466
Activities of daily living          0.749           0.418            0.271                 0.418                  0.084
Urine output                        0.528           0.357            0.407                 0.339                  0.330 

Share with an ADL value, by survival:
died on day 2 to 7      0.033
died on day 8 to 14     0.057
died on day 15 to 30    0.115
alive at day 30         0.345 

Share with a urine output value, by admission year:
1989    0.000
1990    0.000
1991    0.000
1992    0.858
1993    0.852
1994    0.867


The three gaps have different causes.

The secondary disease category only exists for patients who have a second condition from the list. A missing value means none, and these patients die less often, 30.0% against 46.6%. We keep the variable and code the gap as its own category.

Activities of daily living is the opposite case. Among patients who died within the first week almost nobody has a value, among those alive at day 30 about a third do. Whether a value exists depends on how the patient fared after day 1, so the measurement was most likely taken later. We drop the variable and do not add a missing indicator either, because both would bring information from after the treatment decision into the adjustment set.

Urine output was not recorded before 1992 and is available for most patients after that. The gap tells us when a patient was admitted, not how sick the patient was. Filling it in for more than half of the patients from a later period is not credible, so we drop the variable. Whether the admission period itself is a confounder comes up again in Step 2.

In [4]:
df = df_raw.copy()
df["cat2"] = df["cat2"].fillna("None")
DROP_COLS = ["adld3p", "urin1"]
X_COLS = [c for c in df.columns if role(c) == "candidate confounder" and c not in DROP_COLS]

print(f"Covariates: {len(X_COLS)}, missing values left: {df[X_COLS].isna().sum().sum()}")
RESULTS.update({"n_covariates": len(X_COLS)})

Covariates: 51, missing values left: 0


51 covariates remain, none of them with missing values.

---
## Still to do
- 1.4 Analysis sample and unadjusted comparison of 30-day mortality.
- Steps 2 to 8.